# z719 final gerencial: entrega
**Francisco Medina Raed** · Laboratorio 1 · Modalidad Gerencial · Competencia `labo-1-virtual-2026-manager`

Este notebook genera la entrega final: predice qué clientes del banco se van a dar de baja en **septiembre de 2021 (202109)** y arma el archivo para Kaggle con los **1.000 clientes** a contactar.

Parte del workflow **z719** de la cátedra y le aplica los cambios que surgieron del Experimento Colaborativo 07 y de diez corridas propias (WF7190 a WF7199). Más abajo están, en este orden:

1. El problema y cómo se mide.
2. Las diferencias con el workflow original.
3. El feature engineering, explicado bloque por bloque.
4. Los experimentos que llevaron a cada decisión.
5. Qué no funcionó.
6. El código, que reproduce exactamente la predicción subida a Kaggle (WF7198).

## 1. El problema

- El dataset gerencial tiene una fila por cliente y por mes, de 202005 a 202109, con unos 13.200 clientes por mes y **31 variables originales**.
- `clase_ternaria` dice qué pasó después con cada cliente: **CONTINUA**, **BAJA+1** (se va el mes siguiente) o **BAJA+2** (se va dentro de dos meses). Los BAJA+2 son el objetivo, porque hay tiempo para retenerlos. Son unos 85 a 95 por mes: el **0,7 %** de los clientes.
- **Ganancia:** cada BAJA+2 contactado suma **975.000**; cada BAJA+1 o CONTINUA contactado resta **25.000**. Conviene seguir contactando mientras al menos 1 de cada 40 clientes sea BAJA+2.
- El modelo entrena con **BAJA+1 y BAJA+2 juntos como positivos**, para tener más ejemplos de "cliente que se va", y devuelve una probabilidad. Los clientes se ordenan por esa probabilidad y se contacta a los primeros N: el **corte**.
- **Cómo medimos durante el trabajo:** con la ganancia en meses que tienen la clase (sobre todo julio, 202107, con 91 BAJA+2), y con el Public de Kaggle como control. El Public es aproximadamente el 30 % de septiembre (unos 27 BAJA+2), así que un solo cliente lo mueve unos 3,3 puntos: por eso las decisiones técnicas se tomaron en julio y no en el Public.

## 2. Diferencias con el workflow original (z719)

| Etapa | z719 de la cátedra | Esta entrega | Por qué |
|---|---|---|---|
| **Meses de training** | 202005 a 202106 (14 meses, con el ASPO) | 202103 a 202106 para elegir; **202103 a 202107** para el modelo final | Experimento Colaborativo 07 (ver sección 4.1) |
| **Catastrophe analysis** | 12 variables de 202006 a NA | Igual | — |
| **FE dentro del mes** | `kmes` y `mpayroll_sobre_edad` | `mpayroll_sobre_edad` y **6 ratios nuevos** | Sección 3 |
| **Data drifting** | Vacío | **Montos convertidos a ranking dentro de cada mes** | Sección 3 |
| **FE histórico** | Lags y deltas de orden 1 y 2 | Igual, más **tendencias de 6 meses** para 30 variables | Sección 3 |
| **Variables del modelo** | Todas, incluidas `numero_de_cliente` y `foto_mes` | **Sin `numero_de_cliente`, `foto_mes` ni `kmes`** | Sección 3 |
| **Elección de hiperparámetros** | Grid de 20 combinaciones, **por AUC** en 202107, una semilla | Elegidos **por ganancia** en 202107, con 10 a 40 semillas; en este notebook van fijos | Secciones 4.3, 4.5 y 4.6 |
| **Hiperparámetros** | learning rate 0,03; el resto del grid | **learning rate 0,02**, 256 hojas, mínimo 128 por hoja, 150 árboles | WF7193 y WF7197 |
| **Modelo final** | `c(fijos, mejores)`: **el grid search no se aplicaba** (ver 4.2) | Parámetros correctos | WF7190 |
| **Semillas** | Un modelo, una semilla | **Semillerío de 100 modelos**, promediando probabilidades | WF7190 y WF7198 |
| **Cortes** | 800 a 1.300 de a 50, se suben todos | 800 a 1.400 de a 50; **se sube solo 1.000** | WF7199 |

## 3. Feature engineering

El dataset gerencial tiene solo 31 variables originales. Casi todo lo que el modelo sabe sale de lo que se construye con ellas. Este fue el cambio que más ganancia agregó: **+3 M en julio** sobre el modelo sin él, con las 30 combinaciones de hiperparámetros por encima de la mejor anterior, y **+4,6 puntos en el Public** (WF7191 contra WF7192).

El orden importa y es este: ratios → rankings → lags y deltas → tendencias.

### 3.1 Sacar los identificadores
`numero_de_cliente` y `foto_mes` siguen en el dataset (sirven para los lags, para separar meses y para el archivo de Kaggle), pero **no entran al modelo**.
- `numero_de_cliente` es un número de identificación, no un atributo, y aun así estaba en el **puesto 6 a 8 de importancia** en las corridas anteriores: el modelo se apoyaba en él.
- `foto_mes` tiene un problema de fondo: el modelo nunca ve septiembre y los árboles no extrapolan, así que todo corte del tipo "foto_mes > X" manda a septiembre a la misma rama que julio. `kmes` (el mes del año) tiene el mismo problema y no puede captar estacionalidad con pocos meses de training, así que tampoco se crea.

### 3.2 Ratios dentro del mes
Un árbol pregunta por **una variable a la vez**. Si la señal está en la relación entre dos variables, necesita muchas preguntas encadenadas para aproximarla. Darle el cociente ya calculado le sirve la señal en una sola pregunta. Por ejemplo, el mismo saldo puede ser normal para un cliente y una cuenta vaciada para otro, según lo que cobre.

| Variable | Cálculo | Qué busca |
|---|---|---|
| `mpayroll_sobre_edad` | Sueldo / edad | Viene del z719 |
| `mtarjetas_consumo` | Consumo Visa + Master | Actividad total con tarjetas |
| `tarjeta_cerrando` | 1 si alguna tarjeta tiene status 6, 7 o 9 | Tarjeta en cierre o cerrada |
| `tiene_payroll` | 1 si cobra el sueldo en el banco | Su delta = −1 es "dejó de cobrar el sueldo acá" |
| `r_saldo_sobre_payroll` | Saldo / sueldo | Saldo medido en sueldos |
| `r_consumo_sobre_payroll` | Consumo de tarjetas / sueldo | Si gasta poco para lo que gana, usa otro banco |
| `r_trx_por_producto` | Transacciones del trimestre / productos | Muchos productos y poco uso: cliente dormido |

`r_trx_por_producto` quedó en el **puesto 3 de importancia** del modelo. Los ratios se calculan **antes** de los rankings, con los montos en pesos: un cociente entre dos montos del mismo mes no tiene inflación.

### 3.3 Data drifting: rankings por mes
Con la inflación de 2021, un monto en pesos no significa lo mismo en marzo que en septiembre. El modelo aprende cortes del tipo "saldo < X" con los pesos de un mes y los aplica a otro, donde X ya vale otra cosa.

Cada variable de monto (las que empiezan con `m`, `Visa_m` o `Master_m`) se reemplaza por su **posición relativa dentro del mes**: los positivos quedan entre 0 y 1, los negativos entre −1 y 0, y el cero queda en 0, porque "saldo cero" tiene un significado propio. El cliente con más saldo de cada mes vale 1 aunque el monto nominal haya subido. Se hace antes de los lags, así los deltas también quedan sin inflación.

### 3.4 Historia: lags y deltas
Igual que el z719: para cada variable, su valor hace 1 y 2 meses (lag) y la diferencia con el valor actual (delta).

### 3.5 Tendencias en ventana de 6 meses
Quien se va del banco suele mostrar una **caída sostenida respecto de su propia historia**, no solo un cambio de un mes. Para las 30 variables más importantes del WF7191 se calculan, sobre los últimos 6 meses del mismo cliente:

| Variable | Qué mide |
|---|---|
| `_tend6` | Pendiente de una regresión lineal: si viene subiendo o bajando |
| `_avg6` | Promedio |
| `_max6` y `_min6` | Máximo y mínimo |
| `_ratioavg6` | Valor actual / promedio: caída brusca respecto de lo normal para ese cliente |

Si el cliente tiene menos de 6 meses de historia, quedan en NA. Este bloque aporta **el 46 % de la importancia** del modelo; lo que más pesa es el valor actual contra el promedio y los mínimos. Como los lags se calculan sobre todo el dataset antes de elegir meses, estas variables siguen usando información de 2020 aunque el modelo entrene desde 2021.

En total el modelo usa **330 variables**.

## 4. Los experimentos

Cada decisión se tomó con una prueba. Las mediciones en julio usan **varias semillas** y comparaciones **de a pares** (mismas semillas en las dos opciones), porque la ganancia de un solo modelo varía unos 1,1 M según la semilla.

### 4.1 Experimento Colaborativo 07: los meses de pandemia (Grupo A, con Facundo Londero)
**Pregunta:** ¿conviene entrenar con los meses del ASPO o sacarlos?

Se compararon cuatro primeros meses de training con el z610, con 10 semillas pareadas (5 de cada integrante), 6 cortes por corrida y la ganancia como máximo del Public:

| Variante | Desde | Meses | Ganancia media |
|---|---|---|---|
| V0, todos los meses | 202005 | 14 | 20,0 |
| V1, sin ASPO | 202011 | 8 | 21,6 |
| V2, solo 2021 | 202101 | 6 | 24,5 |
| **V3, últimos cuatro meses** | **202103** | **4** | **25,0** |

V3 le ganó a la línea base en **las 10 semillas** (Wilcoxon, p = 0,002) y fue la única diferencia que sobrevivió a la corrección de Holm. Esperábamos que cuatro meses fueran pocos datos, y resultó la mejor: **parecerse al mes que se predice pesa más que la cantidad de datos**. Durante el ASPO se iban unos 44 clientes por mes y en 2021 unos 80: el comportamiento de baja cambió de régimen.

**De ahí sale que esta entrega entrene desde 202103.** En las corridas propias se confirmó: con semillerío, V3 superó a V2 (desde 202101) en los 13 cortes de Kaggle (WF7190).

### 4.2 WF7190: arreglo del modelo final y semillerío
- **Un error del z719:** el modelo final se armaba con `c(fijos, mejores)`. Eso dejaba `num_leaves` y `min_data_in_leaf` duplicados, y LightGBM usaba los primeros (los fijos). **El grid search no tenía efecto.** Se reemplazó por `modifyList`.
- **Semillerío:** cada semilla cambia qué columnas ve cada árbol, y con una sola semilla la ganancia de V3 iba de 19,4 a 30,8 según la semilla. Entrenar varios modelos y promediar sus probabilidades cancela los errores en los que difieren.
- **Lo que no funcionó:** se amplió la grilla con hojas de 16 y 32 clientes. V2 eligió 32 por tener el mejor AUC y su ganancia se desplomó (14,7 de promedio en el Public); restringiendo a 64 o más, volvió a 20,1. Desde entonces, `min_data_in_leaf` arranca en 64.

### 4.3 WF7191: elegir por ganancia y no por AUC
El AUC mide si el modelo ordena bien **toda** la lista y cuenta a los BAJA+1 como aciertos. La ganancia solo mira a los primeros ~1.000 y castiga a los BAJA+1. El AUC falló dos veces: en el Exp. 07 la línea base tenía el mejor AUC y la peor ganancia, y en el WF7190 eligió hojas de 32 clientes. El grid pasó a elegir por **ganancia en 202107**, con early stopping también por ganancia.

### 4.4 WF7192: feature engineering
Todo lo de la sección 3. Ganancia en julio: de 35,7 a **38,7 M**; Public: de 27,0 a **31,7**.

### 4.5 WF7193: más semillas en el grid
Con 3 semillas por combinación, el grid no podía distinguir las mejores. Se pasó a **10 semillas** y se sacaron las combinaciones repetidas (con mínimos por hoja altos, distintos `num_leaves` dan el mismo árbol): 16 combinaciones distintas. Toda la grilla quedó en un rango de **1 M**: `num_leaves` y `min_data_in_leaf` casi no mueven la ganancia. Ganó **256 hojas con mínimo 128**.

### 4.6 WF7196 y WF7197: los demás hiperparámetros
- **WF7196:** se movió un parámetro por vez sobre la base, con 10 semillas: `learning_rate` (0,01 a 0,34), `feature_fraction` (0,1 a 0,9), `max_bin` (15 a 255), `bagging_fraction`, `lambda_l1`, `lambda_l2` y `min_gain_to_split`. **Ninguno mejoró.** Solo empeoraron los extremos: learning rate de 0,05 para arriba, `lambda_l2 = 100` y `max_bin = 15`.
- **WF7197:** `learning_rate = 0.02` había quedado apenas por encima. Se comparó 0,02 contra 0,03 en las cuatro mejores combinaciones, con **40 semillas**: mejora **+0,48 M** en promedio (2,9 errores, a favor en 29 de 40 semillas). Por eso esta entrega usa 0,02 y los 150 árboles que eligió ese experimento.

### 4.7 WF7198: semillerío de 100
Se entrenaron 100 modelos sin julio y se midió en julio el semillerío acumulado, de 1 a 100 semillas:
- Con 10 semillas ya se alcanza casi toda la ganancia; con 20, queda a 0,1 M del nivel final.
- Lo que sigue mejorando es la **estabilidad**: el rango según qué semillas toquen pasa de 2,8 M con una semilla a 0,6 M con 20 y 0,3 M con 50.
- El semillerío aporta en la cabeza de la lista (+1,4 M en 800 envíos) y casi nada en 1.100, donde ya no quedan BAJA+2 para reordenar.

Esta entrega usa **100 semillas** para que el resultado dependa lo menos posible de la suerte.

### 4.8 WF7199: validación a dos meses y elección del corte
Julio marcaba el mejor corte cerca de 800 envíos y el Public cerca de 1.100. Como la entrega predice **dos meses** después del último mes de training y la validación medía a **uno**, se reprodujeron las dos distancias con 5 meses de training, midiendo en junio y en julio (175 BAJA+2):

| Distancia | Corte óptimo | Ganancia | Meseta (a menos de 1 M) |
|---|---|---|---|
| 1 mes | 938 | 32,0 M | 788 a 1.165 |
| **2 meses, como la entrega** | **818** | **30,1 M** | **658 a 1.122** |

- La distancia **no** corre el óptimo hacia 1.100: el pico del Public es más probablemente azar de su muestra.
- Predecir a dos meses cuesta unos 2 M. Eso explica por qué el Public daba alrededor del 80 % de julio, y la estimación de unos 30 M coincide con el Public de los mejores modelos.
- Cortar en 1.000 pierde 0,54 M respecto del óptimo (medio cliente). **Se eligió 1.000** porque queda dentro de la meseta de la validación y lejos de la caída que muestra el Public por debajo de 950.

## 5. Lo que no funcionó

| Prueba | Corrida | Resultado |
|---|---|---|
| Hojas de 16 y 32 clientes | WF7190 | Sobreajuste: la ganancia de V2 se desplomó |
| Diez ratios más (préstamos, homebanking, caja de ahorro, pago mínimo, etc.) | WF7195 | −0,24 M en julio, peor en 13 de 16 combinaciones: repetían información que el modelo ya tenía |
| Selección de variables con canaritos (umbral en la mediana) | WF7194 | Dejó 91 de 380 variables y perdió 1,6 M en julio y 10 puntos en el Public. Con pocos positivos, el azar rinde demasiado bien y el umbral cortó de más |
| Configuración de "modelos débiles" (learning rate 0,34, 32 hojas, feature fraction 0,2) | WF7196 | −5,4 M por modelo y −2,7 M aun en semillerío |
| Otros hiperparámetros | WF7196 | Ninguno mejoró |

**Resumen:** la ganancia la movieron los **meses de training** y el **feature engineering**. Los hiperparámetros están en una meseta: alrededor de la configuración elegida, todo da casi lo mismo.

## 6. El código

### 6.1 Parámetros, paquetes y carpetas
Meses, hiperparámetros fijos (con la corrida de donde sale cada uno), las 100 semillas derivadas de la semilla primigenia y los cortes de Kaggle.

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
rm(list = ls(all.names = TRUE))
invisible(gc(full = TRUE, verbose = FALSE))

for (paquete in c("data.table", "R.utils", "lightgbm", "yaml")) {
  if (!require(paquete, character.only = TRUE)) {
    install.packages(paquete)
    require(paquete, character.only = TRUE)
  }
}

PARAM <- list()
PARAM$experimento <- 7198
PARAM$semilla_primigenia <- 123457
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# --- Meses ---
PARAM$final_train <- c(202103, 202104, 202105, 202106, 202107)   # Exp. Colaborativo 07: entrenar desde 202103
PARAM$future <- 202109
PARAM$positivos <- c("BAJA+1", "BAJA+2")

# --- Hiperparametros de LightGBM (fijos: no se hace grid search) ---
# num_leaves y min_data_in_leaf: grid del WF7193 (16 combinaciones, 10 semillas)
# learning_rate y num_iterations: WF7197 (0.02 mejoro a 0.03 en 4 combinaciones, 40 semillas; 150 arboles)
# feature_fraction, max_bin y el resto: los del z719; el barrido del WF7196 no encontro mejoras
PARAM$lgbm <- list(
  objective = "binary",
  metric = "None",
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = PARAM$semilla_primigenia,     # se reemplaza por cada semilla del semillerio
  max_bin = 31,
  learning_rate = 0.02,
  feature_fraction = 0.5,
  num_leaves = 256,
  min_data_in_leaf = 128,
  num_iterations = 150
)

# --- Semillerio: 100 modelos, cada uno con una semilla derivada de la primigenia ---
PARAM$semillerio$cantidad <- 100
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillerio$semillas <- c(PARAM$semilla_primigenia,
  setdiff(sample(100000:999999, PARAM$semillerio$cantidad + 5), PARAM$semilla_primigenia)[seq_len(PARAM$semillerio$cantidad - 1)])

# --- Kaggle ---
PARAM$kaggle$competencia <- "labo-1-virtual-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1400, by = 50)   # se generan todos los archivos, para poder reproducirlos
PARAM$kaggle$subir  <- c(1000)                   # se sube solo la entrega; agregar 900 para la segunda opcion
subir_kaggle <- TRUE

# --- Carpetas ---
base <- path.expand(if (dir.exists("~/buckets/b1")) "~/buckets/b1/exp" else "~/exp")
carpeta <- file.path(base, paste0("WF", PARAM$experimento, "_final"))
dir.create(carpeta, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta)

candidatos <- path.expand(file.path(c("~/datasets", "~/buckets/b1/datasets", carpeta), PARAM$dataset))
archivo_dataset <- candidatos[file.exists(candidatos)][1]
if (is.na(archivo_dataset)) {
  archivo_dataset <- file.path(carpeta, PARAM$dataset)
  download.file(paste0("https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/", PARAM$dataset),
                archivo_dataset, mode = "wb")
}
message("carpeta: ", carpeta, "\ndataset: ", archivo_dataset)

### 6.2 Preprocesamiento y feature engineering
Catastrophe analysis, ratios, rankings por mes, lags y deltas, tendencias y variables del modelo, en el orden de la sección 3.

In [ ]:
dataset <- fread(archivo_dataset)

# ---------- Catastrophe Analysis: variables que en 202006 vinieron todas en cero ----------
for (v in c("internet", "mrentabilidad", "mrentabilidad_annual", "mcomisiones", "mactivos_margen",
            "mpasivos_margen", "mcuentas_saldo", "ctarjeta_visa_transacciones", "mtarjeta_visa_consumo",
            "mtarjeta_master_consumo", "ccallcenter_transacciones", "chomebanking_transacciones"))
  dataset[foto_mes == 202006, (v) := NA]

# ---------- Feature engineering dentro del mes ----------
dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# suma por fila que da NA solo si todos los sumandos son NA
suma_na <- function(...) {
  m <- cbind(...)
  s <- rowSums(m, na.rm = TRUE)
  s[rowSums(!is.na(m)) == 0] <- NA
  s
}
# ratios (se calculan antes de los rankings, sobre montos nominales del mismo mes)
dataset[, mtarjetas_consumo := suma_na(mtarjeta_visa_consumo, mtarjeta_master_consumo)]
dataset[, tarjeta_cerrando := as.integer(Visa_status %in% c(6, 7, 9) | Master_status %in% c(6, 7, 9))]
dataset[, tiene_payroll := as.integer(mpayroll > 0)]
dataset[, r_saldo_sobre_payroll := ifelse(mpayroll > 0, mcuentas_saldo / mpayroll, NA_real_)]
dataset[, r_consumo_sobre_payroll := ifelse(mpayroll > 0, mtarjetas_consumo / mpayroll, NA_real_)]
dataset[, r_trx_por_producto := ctrx_quarter / cproductos]

# ---------- Data drifting: cada monto pasa a su ranking dentro del mes ----------
# positivos en (0, 1], negativos en [-1, 0), el cero queda en 0 y los NA quedan NA
rank_cero_fijo <- function(x) {
  r <- rep(NA_real_, length(x))
  pos <- which(x > 0); neg <- which(x < 0)
  r[which(x == 0)] <- 0
  if (length(pos)) r[pos] <- frank(x[pos], ties.method = "average") / length(pos)
  if (length(neg)) r[neg] <- -frank(-x[neg], ties.method = "average") / length(neg)
  r
}
cols_monetarias <- grep("^(m|Visa_m|Master_m)", colnames(dataset), value = TRUE)
for (v in cols_monetarias) set(dataset, j = v, value = as.numeric(dataset[[v]]))
dataset[, (cols_monetarias) := lapply(.SD, rank_cero_fijo), by = foto_mes, .SDcols = cols_monetarias]

# ---------- Feature engineering historico: lags y deltas de orden 1 y 2 ----------
stopifnot(dataset[, all(diff(foto_mes) > 0), by = numero_de_cliente][, all(V1)])   # orden cronologico por cliente
cols_lagueables <- copy(setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria")))
dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"), by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"), by = numero_de_cliente, .SDcols = cols_lagueables]
for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# ---------- Tendencias en ventana de 6 meses ----------
# las 30 variables mas importantes del WF7191 (la lista que usaron el WF7192 y siguientes)
cols_tendencia <- c("ctrx_quarter", "mcaja_ahorro", "mpayroll", "mcuenta_corriente", "cliente_edad",
  "Visa_mpagominimo", "cpayroll_trx", "mtarjeta_visa_consumo", "mcuentas_saldo", "mprestamos_personales",
  "mpasivos_margen", "mrentabilidad_annual", "cproductos", "Visa_fechaalta", "Master_fechaalta", "mcomisiones",
  "mpayroll_sobre_edad", "mactivos_margen", "cliente_antiguedad", "mrentabilidad", "chomebanking_transacciones",
  "ctarjeta_visa_transacciones", "ccomisiones_mantenimiento", "Visa_status", "cdescubierto_preacordado",
  "mtarjeta_master_consumo", "Master_mpagominimo", "ccallcenter_transacciones", "Master_status", "internet")
n <- 6
ord <- order(dataset$numero_de_cliente, dataset$foto_mes)          # se trabaja ordenado por cliente y mes
pos <- sequence(rle(dataset$numero_de_cliente[ord])$lengths)       # numero de mes dentro de cada cliente
incompleta <- pos < n                                               # la ventana mezclaria dos clientes
w <- (1:n) - (n + 1) / 2                                            # pesos de la pendiente de minimos cuadrados
den <- sum(w^2)
guardar <- function(nombre, valores) {
  valores[incompleta] <- NA
  res <- rep(NA_real_, length(ord))
  res[ord] <- valores                                               # vuelve al orden original de las filas
  set(dataset, j = nombre, value = res)
}
for (v in cols_tendencia) {
  x <- as.numeric(dataset[[v]][ord])
  pendiente <- Reduce(`+`, lapply(1:n, function(j) w[j] * shift(x, n - j))) / den
  promedio <- frollmean(x, n)
  guardar(paste0(v, "_tend", n), pendiente)                         # pendiente de los ultimos 6 meses
  guardar(paste0(v, "_avg", n), promedio)                           # promedio
  guardar(paste0(v, "_max", n), frollapply(x, n, max))              # maximo
  guardar(paste0(v, "_min", n), frollapply(x, n, min))              # minimo
  guardar(paste0(v, "_ratioavg", n), x / promedio)                  # valor actual contra su promedio
}

# ---------- Clase y variables del modelo ----------
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$positivos, 1L, 0L)]
# numero_de_cliente y foto_mes quedan en el dataset (para los lags, los filtros y Kaggle) pero no entran al modelo
campos_buenos <- setdiff(colnames(dataset), c("clase_ternaria", "clase01", "numero_de_cliente", "foto_mes"))
# los cocientes pueden dar Inf o NaN: pasan a NA
for (j in campos_buenos) if (is.double(dataset[[j]])) {
  malos <- which(is.nan(dataset[[j]]) | is.infinite(dataset[[j]]))
  if (length(malos)) set(dataset, i = malos, j = j, value = NA_real_)
}
message(format(Sys.time(), "%a %b %d %X %Y"), "  preprocesado: ", nrow(dataset), " filas, ",
        length(campos_buenos), " variables del modelo")

### 6.3 Semillerío de 100 y predicción de septiembre
Cada modelo entrena con 202103 a 202107 y predice 202109. La predicción de cada semilla se guarda en `semillas/` y la probabilidad final es el promedio de las 100. Tarda unos 10 a 15 minutos.

In [ ]:
# ---------- Semillerio: 100 modelos con 202103-202107, cada uno predice 202109 ----------
filtro_train <- dataset[, foto_mes %in% PARAM$final_train]
dtrain <- lgb.Dataset(data = data.matrix(dataset[filtro_train, campos_buenos, with = FALSE]),
                      label = dataset[filtro_train, clase01], free_raw_data = TRUE)
dfuture <- dataset[foto_mes == PARAM$future]
mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

dir.create("semillas", showWarnings = FALSE)
param <- PARAM$lgbm
t0 <- Sys.time()
for (i in seq_along(PARAM$semillerio$semillas)) {
  param$seed <- PARAM$semillerio$semillas[i]
  modelo <- lgb.train(data = dtrain, param = param, verbose = -100)
  # cada prediccion se guarda en su archivo y despues se promedian los archivos, igual que en el WF7198
  fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, prob = predict(modelo, mfuture)),
         file.path("semillas", paste0("pred_s", param$seed, ".txt")), sep = "\t")
  if (i == 1) {                                         # modelo e importancia de la semilla primigenia
    lgb.save(modelo, "modelo.txt")
    fwrite(as.data.table(lgb.importance(modelo)), "impo.txt", sep = "\t")
  }
  if (i %% 10 == 0) message(format(Sys.time(), "%a %b %d %X %Y  "), i, " de ", length(PARAM$semillerio$semillas), " semillas")
  rm(modelo); invisible(gc(full = TRUE, verbose = FALSE))
}

# probabilidad final: promedio de las probabilidades de los 100 modelos
preds <- rbindlist(lapply(PARAM$semillerio$semillas, function(s) fread(file.path("semillas", paste0("pred_s", s, ".txt")))))
tb_prediccion <- preds[, .(prob = mean(prob)), by = numero_de_cliente]
fwrite(tb_prediccion, "prediccion.txt", sep = "\t")
message("semillerio listo en ", round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " minutos")

### 6.4 Archivos para Kaggle, subida y registro
Ordena a los clientes por probabilidad, genera un archivo por corte (800 a 1.400 de a 50) y sube solo el de 1.000 envíos. Si el archivo ya está en Kaggle, poner `subir_kaggle <- FALSE` en la primera celda. Guarda `PARAM.yml` con toda la configuración.

In [ ]:
# ---------- Archivos para Kaggle ----------
dir.create("kaggle", showWarnings = FALSE)
tb <- copy(tb_prediccion)
setorder(tb, -prob)
for (envios in PARAM$kaggle$cortes) {
  tb[, Predicted := 0L]
  tb[1:envios, Predicted := 1L]
  fwrite(tb[, .(numero_de_cliente, Predicted)],
         file.path("kaggle", paste0("KA", PARAM$experimento, "_final_", envios, ".csv")), sep = ",")
}
message(length(PARAM$kaggle$cortes), " archivos generados en ", file.path(carpeta, "kaggle"))

# ---------- Subida de la entrega ----------
if (subir_kaggle) {
  if (!file.exists("~/.kaggle/kaggle.json")) stop("falta ~/.kaggle/kaggle.json en la VM")
  if (Sys.which("kaggle") == "") invisible(system("python3 -m pip install -q --user kaggle"))
  for (envios in PARAM$kaggle$subir) {
    archivo <- file.path("kaggle", paste0("KA", PARAM$experimento, "_final_", envios, ".csv"))
    mensaje <- paste0("z719 final ", PARAM$experimento, " semillerio=", PARAM$semillerio$cantidad,
                      " semilla=", PARAM$semilla_primigenia, " envios=", envios)
    salida <- system2("kaggle", c("competitions", "submit", "-c", PARAM$kaggle$competencia,
                                  "-f", shQuote(archivo), "-m", shQuote(mensaje)), stdout = TRUE, stderr = TRUE)
    message(basename(archivo), ": ", paste(trimws(salida), collapse = " "))
  }
}

# ---------- Registro de la corrida ----------
PARAM$out$variables_modelo <- length(campos_buenos)
PARAM$out$filas_final_train <- sum(filtro_train)
write_yaml(PARAM, "PARAM.yml")
format(Sys.time(), "%a %b %d %X %Y")

### 6.5 Verificación (opcional)
Si encuentra la carpeta del WF7198 original, compara las dos predicciones. Deben ser idénticas.

In [ ]:
# ---------- Verificacion (opcional): ¿reproduce la prediccion del WF7198? ----------
original <- file.path(base, "WF7198", "V3fe_noid_rank_ratio_tend_sem100", "prediccion.txt")
if (!file.exists(original)) {
  message("No encuentro ", original, ": no se puede comparar")
} else {
  orig <- fread(original)
  nueva <- fread("prediccion.txt")                 # se comparan los dos archivos, tal como quedaron en disco
  comp <- merge(nueva, orig, by = "numero_de_cliente", suffixes = c("_nueva", "_7198"))
  dif_max <- max(abs(comp$prob_nueva - comp$prob_7198))
  top <- function(x, k) x[order(-prob)][1:k, numero_de_cliente]
  iguales <- sapply(PARAM$kaggle$subir, function(k)
    setequal(top(nueva, k), top(orig, k)))
  message("Clientes comparados: ", nrow(comp), " de ", nrow(nueva),
          ".  Diferencia maxima de probabilidad: ", signif(dif_max, 3))
  message("Mismos clientes elegidos en ", paste(PARAM$kaggle$subir, collapse = ", "), " envios: ",
          paste(iguales, collapse = ", "))
  if (dif_max == 0) message("REPRODUCE EXACTAMENTE la prediccion del WF7198") else
    message("No es identica: revisar version de LightGBM, dataset o semillas")
}